# Cluster count analysis

CyteType is billed per cluster, so this notebook tallies the cluster counts in `output/clustering_pipeline/<run>/run.csv` and then breaks them down per disease label. Two views are produced:

- An inclusive view that uses every matching `DISEASE_MAP` label (parent and child overlap, useful for the lung-cancer hierarchy).
- A disjoint view that assigns each accession to its single most-specific label (and `Other` for anything unmatched), so per-label sums add up to the true total.

Inputs:

- `output/clustering_pipeline/<run>/run.csv` (clustering pipeline summary, one row per accession)
- `output/metadata/accession_disease_categories.json` (per-accession `DISEASE_MAP` labels, built by `notebooks/pipeline/metadata.ipynb`)
- `output/metadata/datasets_subset_qc.csv` (per-cohort cytetype QC subset, also built by `notebooks/pipeline/metadata.ipynb`)

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from metadata import most_specific_disease_label
from shared.repo import REPO_ROOT

In [ ]:
# Inspect columns
df = pd.read_csv(REPO_ROOT / "output/clustering_pipeline/20260519_160627/run.csv")
df.columns

In [ ]:
# Total number of clusters across data
df["nClustersPostMerge"].sum()

In [ ]:
# Number of unique cluster counts
nclust = df["nClustersPostMerge"]
np.unique(nclust)

In [ ]:
# Filter dataframe, show total number of clusters after filtering
df = df[df["nClustersPostMerge"] < 100]
df = df.dropna(subset=["nClustersPostMerge"])  # pyright: ignore[reportCallIssue]
df["nClustersPostMerge"].sum()

## Disease categories per accession

Load the JSON produced by `notebooks/pipeline/metadata.ipynb` (`output/metadata/accession_disease_categories.json`). Each accession maps to its raw `disease` string and the list of `DISEASE_MAP` labels matched by its disease string. Multiple labels are expected for the nested lung-cancer subtree (parent `Lung Cancer` plus a child like `NSCLC` and a grandchild like `LUAD`); cross-cohort overlap between the sibling cohort labels (`IPF`, `COVID-19`, `COPD`, `Cystic Fibrosis`, `Interstitial Lung Disease`, `Pulmonary Hypertension`) is rare (~5 rows of ~800). Accessions whose disease string is `non-cystic fibrosis` / `non-CF` (pure or mixed) are stripped of the CF label inside `disease_categories_for`, so they show up here with an empty list and end up in `Other` in the disjoint view below.

In [ ]:
import json

categories_path = REPO_ROOT / "output/metadata/accession_disease_categories.json"
with categories_path.open() as f:
    accession_categories = json.load(f)

categories_df = pd.DataFrame(
    [
        {"srx_accession": srx, "disease": entry["disease"], "categories": entry["categories"]}
        for srx, entry in accession_categories.items()
    ]
)
print(f"Loaded {len(categories_df):,} accessions from {categories_path}")
categories_df.head()

In [ ]:
# Frequency of each label across accessions (counts every match, so an accession
# that hit Lung Cancer + NSCLC + LUAD contributes to all three).
label_counts = (
    categories_df.explode("categories")
    .dropna(subset=["categories"])
    .groupby("categories")
    .size()
    .sort_values(ascending=False)
)
label_counts

In [ ]:
# Join categories onto the clustering run frame and show how many run.csv accessions
# have a category mapping at all (the JSON covers only the lung intersection).
clusters_with_categories = df.merge(
    categories_df, left_on="srx", right_on="srx_accession", how="left"
)
n_mapped = clusters_with_categories["categories"].notna().sum()
print(f"{n_mapped:,} of {len(clusters_with_categories):,} run.csv rows have a disease-category entry")
clusters_with_categories[["srx", "nClustersPostMerge", "disease", "categories"]].head()

In [ ]:
# Sum of nClustersPostMerge per disease label. An accession with multiple labels
# (e.g. Lung Cancer + NSCLC + LUAD) contributes its cluster count to each of them.
clusters_per_label = (  # pyright: ignore[reportCallIssue]
    clusters_with_categories.dropna(subset=["categories", "nClustersPostMerge"])
    .explode("categories")
    .groupby("categories")["nClustersPostMerge"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)
clusters_per_label

### Disjoint partition (most-specific label wins)

Each accession is assigned to a single bucket using `most_specific_disease_label` from `metadata.categorize`, which picks `cats[-1]` from `disease_categories_for`. Inside the lung-cancer subtree this is the most-specific child by design (`DISEASE_MAP` lists parents before children). For the few cross-cohort comorbidities (e.g. `lung cancer, COPD`) the tie-break is just the later sibling in `DISEASE_MAP` order; see the corresponding note in `notebooks/pipeline/metadata.ipynb`. Accessions whose categories list is empty (`Other`) include unmatched diseases and the `non-CF` rows. `plot_disease_breakdown` in `scripts/metadata/viz.py` uses the same helper.

In this view every accession contributes its cluster count exactly once, so columns are non-overlapping and add up to a true total.

In [ ]:
_ready = clusters_with_categories.dropna(subset=["disease", "nClustersPostMerge"]).copy()
_ready["mostSpecificLabel"] = _ready["disease"].map(most_specific_disease_label)

clusters_per_label_disjoint = (
    _ready.groupby("mostSpecificLabel")["nClustersPostMerge"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)
clusters_per_label_disjoint

In [ ]:
label_comparison = (
    pd.concat(
        [clusters_per_label.rename("inclusive"), clusters_per_label_disjoint.rename("disjoint")],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
label_comparison["overlapAbsorbed"] = label_comparison["inclusive"] - label_comparison["disjoint"]
label_comparison.sort_values("inclusive", ascending=False)

## Cluster counts in the cytetype QC dataset

Load `datasets_subset_qc.csv` (built by `notebooks/pipeline/metadata.ipynb`), intersect it with `run.csv`, and inspect cluster counts per disease label. The CSV is already disjoint by construction: each accession has exactly one `diseaseLabel` drawn from `IPF / Pulmonary Fibrosis`, `COVID-19 / SARS-CoV-2`, `COPD`, `Interstitial Lung Disease`, or `Cystic Fibrosis`, after per-label QC and bounded sampling. Per-label sums therefore add up cleanly to a single total.

In [ ]:
datasets_subset_qc_path = REPO_ROOT / "output/metadata/datasets_subset_qc.csv"
subset_df = pd.read_csv(datasets_subset_qc_path)
clusters_subset = df.merge(subset_df, left_on="srx", right_on="srx_accession", how="inner")
print(f"{len(clusters_subset):,} of {len(subset_df):,} datasets_subset_qc accessions present in run.csv")
clusters_subset.head()

In [ ]:
subset_clusters_per_label = (
    clusters_subset.groupby("diseaseLabel")["nClustersPostMerge"]
    .agg(nAccessions="count", nClusters="sum")
    .astype(int)
    .sort_values("nClusters", ascending=False)
)
total_clusters = int(clusters_subset["nClustersPostMerge"].sum())
total_accessions = len(clusters_subset)
print(f"total: {total_clusters:,} clusters across {total_accessions:,} accessions")
subset_clusters_per_label